# Azure Blob Storage Setup


In [ ]:
# Note - you have to login via 'azure login'
# Install in case you have no downloaded requirements.txt

# !pip install azure-storage-blob azure-identity


In [ ]:
# Add Auth by Get your signed-in user email/ID
# Assign the data role to yourself
# az role assignment create \
#   --role "Storage Blob Data Contributor" \
#    --assignee "$USER_EMAIL" \
#    --scope "/subscriptions/a85a22ed-4fbf-4d3c-bf64-3c130639d180/resourceGroups/rg-mlops-test/providers/Microsoft.Storage/storageAccounts/your_storage_account_name"

In [ ]:

# import platform
# from dotenv import load_dotenv
import os
import subprocess
import time
from azure.identity import DefaultAzureCredential
from azure.mgmt.storage import StorageManagementClient
from azure.storage.blob import BlobServiceClient
from azure.mgmt.resource import ResourceManagementClient
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import SubscriptionClient
from azure.identity import SharedTokenCacheCredential, DefaultAzureCredential


In [ ]:
# Load .env if it exists
# load_dotenv()

In [19]:
# 1. Setup Credential
# This will find your 'az login' tokens directly from the file system 
# even if the 'az' command itself isn't in the PATH.
credential = DefaultAzureCredential()

try:
    # 2. Fetch Subscription ID via SDK
    sub_client = SubscriptionClient(credential)
    # Get the first active subscription
    sub_list = list(sub_client.subscriptions.list())
    if not sub_list:
        raise Exception("No active subscriptions found. Run 'az login' in your terminal.")
    
    # Use the first one or filter by a name if you prefer
    SUBSCRIPTION_ID = sub_list[0].subscription_id
    
    # 3. Define Global Variables
    RESOURCE_GROUP = "rg-mlops-test"
    LOCATION = "eastus"
    CONTAINER_NAME = "mlops-data"
    
    # Use a clean slice of the sub ID for uniqueness
    STORAGE_ACCOUNT_NAME = f"mlopstest{SUBSCRIPTION_ID.split('-')[0]}" 

    print(f"✅ Environment Ready (SDK Mode)")
    print(f"🌍 Sub ID: {SUBSCRIPTION_ID}")
    print(f"📦 Storage Name: {STORAGE_ACCOUNT_NAME}")

except Exception as e:
    print(f"❌ Failed to initialize: {e}")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

❌ Failed to initialize: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'a

In [16]:

# Define Global Variables

RESOURCE_GROUP = "rg-mlops-test"
LOCATION = "eastus"
CONTAINER_NAME = "mlops-data"

# Fetch Subscription ID dynamically from your active CLI session
SUBSCRIPTION_ID = subprocess.check_output("az account show --query id -o tsv", shell=True, text=True).strip()
STORAGE_ACCOUNT_NAME = f"mlopstest{SUBSCRIPTION_ID[:5]}" # Ensures a unique name

print(f"✅ Environment Ready.\n🌍 Sub: {SUBSCRIPTION_ID}\n📦 Storage Name: {STORAGE_ACCOUNT_NAME}")

/bin/sh: az: command not found


CalledProcessError: Command 'az account show --query id -o tsv' returned non-zero exit status 127.

In [5]:
try:
    # This will now skip environment variables and use your 'az login'
    credential = DefaultAzureCredential()
    
    # Resource Managers (Management Plane)
    resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
    storage_mgmt_client = StorageManagementClient(credential, SUBSCRIPTION_ID)
    
    print("✅ Authentication successful (using Azure CLI session).")
except Exception as e:
    print(f"❌ Authentication failed: {e}")

❌ Authentication failed: name 'SUBSCRIPTION_ID' is not defined


In [ ]:
# creation of the Resource Group and the Storage Account

print(f"🛠️ Ensuring Resource Group '{RESOURCE_GROUP}'...")
resource_client.resource_groups.create_or_update(RESOURCE_GROUP, {"location": LOCATION})

print(f"🛠️ Provisioning Storage Account '{STORAGE_ACCOUNT_NAME}'...")
# This may take 30-60 seconds
storage_poller = storage_mgmt_client.storage_accounts.begin_create(
    RESOURCE_GROUP, 
    STORAGE_ACCOUNT_NAME,
    {
        "location": LOCATION,
        "sku": {"name": "Standard_LRS"},
        "kind": "StorageV2"
    }
)
account_result = storage_poller.result()
primary_endpoint = account_result.primary_endpoints.blob

print(f"✅ Storage Account is LIVE at: {primary_endpoint}")

In [ ]:
def ensure_container(service_client, container_name):
    """
    Checks if a container exists; if not, creates it.
    Returns the container_client.
    """
    container_client = service_client.get_container_client(container_name)
    try:
        container_client.create_container()
        print(f"✅ Container '{container_name}' created successfully.")
    except ResourceExistsError:
        print(f"ℹ️ Container '{container_name}' already exists. Skipping creation.")
    except Exception as e:
        print(f"❌ Failed to ensure container: {e}")
        raise
    return container_client

In [ ]:
def upload_blob(container_client, local_path, blob_name):
    """
    Uploads a local file to the specified container.
    """
    try:
        with open(local_path, "rb") as data:
            container_client.upload_blob(name=blob_name, data=data, overwrite=True)
        print(f"✅ Uploaded '{local_path}' to '{blob_name}'.")
    except FileNotFoundError:
        print(f"❌ Local file not found: {local_path}")
    except Exception as e:
        print(f"❌ Upload failed: {e}")

In [ ]:
def list_my_blobs(container_client):
    """
    Lists all blobs in the container.
    """
    print(f"📋 Blobs in '{container_client.container_name}':")
    blobs = container_client.list_blobs()
    for b in blobs:
        print(f" - {b.name} ({b.size} bytes)")

In [ ]:
credential = DefaultAzureCredential(exclude_shared_token_cache_credential=True)

# 2. Re-initialize the Service Client
blob_service_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net", 
    credential=credential
)

print("🔄 Token refresh triggered. Re-attempting Data Plane connection...")

# 3. Validation Test
try:
    # A simple call to list containers to see if the 'Mismatch' is gone
    containers = list(blob_service_client.list_containers())
    print(f"✅ Access Verified! Found {len(containers)} containers.")
except Exception as e:
    if "AuthorizationPermissionMismatch" in str(e):
        print("⏳ Still waiting for RBAC propagation... Azure can take 1-5 minutes.")
        print("💡 Pro-Tip: If this persists, run 'az logout' and 'az login' in your terminal.")
    else:
        print(f"❌ Connection Error: {e}")

In [ ]:
# Get the key via CLI
keys_cmd = f"az storage account keys list -g {RESOURCE_GROUP} -n {STORAGE_ACCOUNT_NAME} --query '[0].value' -o tsv"
account_key = subprocess.check_output(keys_cmd, shell=True, text=True).strip()

# Connect using the Key (No RBAC required)
blob_service_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net", 
    credential=account_key
)
print("🔑 Connected via Account Key (RBAC Bypassed).")

In [ ]:
def download_blob(blob_name, download_file_path):
    """
    Downloads a blob from Azure to a local path.
    Handles directory creation and provides clear status updates.
    """
    try:
        print(f"🔍 Locating blob: {blob_name}...")
        
        # 1. Get the blob client
        blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=blob_name)
        
        # 2. Ensure the local directory exists (prevents FileNotFoundError)
        local_dir = os.path.dirname(download_file_path)
        if local_dir and not os.path.exists(local_dir):
            os.makedirs(local_dir)
            print(f"📁 Created local directory: {local_dir}")

        # 3. Perform the download
        print(f"⬇️  Downloading '{blob_name}' to '{download_file_path}'...")
        with open(download_file_path, 'wb') as file:
            data = blob_client.download_blob()
            file.write(data.readall())
            
        print(f"✅ Downloaded successfully: {os.path.abspath(download_file_path)}")

    except ResourceNotFoundError:
        print(f"❌ Error: The blob '{blob_name}' was not found in container '{CONTAINER_NAME}'.")
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")

# --- Test Execution ---
download_blob('somefile.html', 'data_download/index.html')